# 21b — Submit GNN Training Job

Submit the GNN-SDM training as a SageMaker Training Job.
Supports two modes:
- **Selected species** (12 species, quick validation)
- **All species** (3,756 species, production run)

Uses spot instances for ~70% cost savings on the production run.
Checkpointing ensures no work is lost if the spot instance is interrupted.

**Outputs** (saved to S3):
- Suitability scores for all patches (`suitability_scores.npz`)
- Training results CSV with per-species metrics
- Per-species model weights (selected species mode only)

### 1. Upload training data to S3

In [1]:
import config
import s3fs

s3 = s3fs.S3FileSystem(anon=False)
training_data_s3 = config.S3_PROCESSED + '/gnn_training_input'

# Upload the files the training script needs
files_to_upload = [
    'landscape_graph.pkl',
    'patch_features.npy',
    'species_patches.pkl',
]

for fname in files_to_upload:
    s3_path = f'{training_data_s3}/{fname}'
    if not s3.exists(s3_path):
        print(f'Uploading {fname}...')
        s3.put(fname, s3_path)
    else:
        print(f'Already exists: {fname}')

print(f'\nTraining data at: {training_data_s3}')

Already exists: landscape_graph.pkl
Already exists: patch_features.npy
Already exists: species_patches.pkl

Training data at: s3://km-cas-datalake/processed/gnn_training_input


### 2. Configure and submit job

Set `MODE` to choose between:
- `"selected"` — 12 species, ~3 min on GPU (for testing)
- `"production"` — all species, ~17 hours on GPU with spot instances

In [2]:
import sagemaker
from sagemaker.pytorch import PyTorch
import boto3

session = sagemaker.Session()
role = sagemaker.get_execution_role()
sm_client = boto3.client('sagemaker')

# ===== CONFIGURATION =====
MODE = 'production'  # 'selected' or 'production'
FORCE_RESUBMIT = False  # Set True to submit even if a completed job exists
# =========================

JOB_NAME_PREFIX = f'gnn-sdm-{MODE}'

print(f'Mode:   {MODE}')
print(f'Role:   {role}')
print(f'Region: {session.boto_region_name}')

sagemaker.config INFO - Fetched defaults config from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
Mode:   production
Role:   arn:aws:iam::666839000341:role/datazone_usr_role_4nntiyjwycnax7_4mc0gojpuy1fnv
Region: e

In [3]:
# Check for existing jobs
in_progress_jobs = sm_client.list_training_jobs(
    StatusEquals='InProgress',
    SortBy='CreationTime',
    SortOrder='Descending',
    MaxResults=10,
)

completed_jobs = sm_client.list_training_jobs(
    NameContains=JOB_NAME_PREFIX,
    StatusEquals='Completed',
    SortBy='CreationTime',
    SortOrder='Descending',
    MaxResults=5,
)

active_job = None
completed_job = None

for job in in_progress_jobs.get('TrainingJobSummaries', []):
    active_job = job
    break

for job in completed_jobs.get('TrainingJobSummaries', []):
    completed_job = job
    break

JOB_NAME = None

if active_job:
    JOB_NAME = active_job['TrainingJobName']
    print(f'\n\u26a0\ufe0f  Job already running: {JOB_NAME}')
    print(f'   Status: {active_job["TrainingJobStatus"]}')
    print(f'   Started: {active_job["CreationTime"]}')
    print('\n   Skip to cell 3 to monitor progress.')
elif completed_job and not FORCE_RESUBMIT:
    JOB_NAME = completed_job['TrainingJobName']
    print(f'\n\u2705 Previous job completed: {JOB_NAME}')
    print(f'   Finished: {completed_job.get("TrainingEndTime", "unknown")}')
    print('\n   Skip to cell 4 to download results.')
    print('   Set FORCE_RESUBMIT = True above to run again.')
else:
    print('\nNo active/completed jobs. Ready to submit.')


✅ Previous job completed: gnn-sdm-production-2026-05-05-06-32-21-658
   Finished: 2026-05-05 19:43:15.542000+00:00

   Skip to cell 4 to download results.
   Set FORCE_RESUBMIT = True above to run again.


In [9]:
# Submit job
if active_job:
    print('Job is already running — not submitting a new one.')
elif JOB_NAME and not FORCE_RESUBMIT:
    print(f'Using existing completed job: {JOB_NAME}')
else:
    # Job configuration per mode
    if MODE == 'selected':
        job_config = dict(
            entry_point='train_selected_species.py',
            instance_type='ml.g4dn.xlarge',
            max_run=3600 * 2,
            use_spot_instances=False,
            hyperparameters={},
            checkpoint_s3_uri=None,
        )
    else:  # production
        # At ~16s/species on GPU, 3756 species ≈ 17 hours
        # Spot instances save ~70% cost
        # Checkpointing ensures progress survives interruptions
        checkpoint_s3 = config.S3_PROCESSED + '/gnn_production_checkpoints'
        job_config = dict(
            entry_point='train_all_species.py',
            instance_type='ml.g4dn.xlarge',
            max_run=3600 * 24,       # 24h max
            use_spot_instances=True,
            max_wait=3600 * 36,      # willing to wait up to 36h total (with interruptions)
            hyperparameters={
                'epochs': 500,
                'patience': 50,
                'hidden-dims': '64,48,32',
                'min-patches': 5,
                'checkpoint-every': 50,
            },
            checkpoint_s3_uri=checkpoint_s3,
        )

    estimator = PyTorch(
        entry_point=job_config['entry_point'],
        source_dir='training_job',
        role=role,
        instance_count=1,
        instance_type=job_config['instance_type'],
        framework_version='2.1',
        py_version='py310',
        output_path=config.S3_PROCESSED + '/gnn_training_output',
        hyperparameters=job_config.get('hyperparameters', {}),
        max_run=job_config['max_run'],
        use_spot_instances=job_config.get('use_spot_instances', False),
        max_wait=job_config.get('max_wait'),
        checkpoint_s3_uri=job_config.get('checkpoint_s3_uri'),
        base_job_name=JOB_NAME_PREFIX,
    )

    estimator.fit(
        inputs={'training': training_data_s3},
        wait=False,
    )

    JOB_NAME = estimator.latest_training_job.name
    print(f'\n\U0001f680 Job submitted: {JOB_NAME}')
    print(f'   Instance: {job_config["instance_type"]}')
    print(f'   Spot: {job_config.get("use_spot_instances", False)}')
    print(f'   Max run: {job_config["max_run"] / 3600:.0f}h')
    print('\n   Monitor in SageMaker Console \u2192 Training \u2192 Training jobs')

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.VpcConfig.Subnets
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.VpcConfig.SecurityGroupIds
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment

🚀 Job submitted: gnn-sdm-production-2026-05-05-06-32-21-658
   Instance: ml.g4dn.xlarge
   Spot: True
   Max run: 24h

   Monitor in SageMaker Console → Training → Training jobs


### 3. Check job status

In [4]:
desc = sm_client.describe_training_job(TrainingJobName=JOB_NAME)
status = desc['TrainingJobStatus']
print(f'Job:    {JOB_NAME}')
print(f'Status: {status}')

if status == 'InProgress':
    secondary = desc.get('SecondaryStatus', '')
    print(f'Phase:  {secondary}')
    start = desc.get('TrainingStartTime')
    if start:
        from datetime import datetime, timezone
        elapsed = datetime.now(timezone.utc) - start
        print(f'Elapsed: {elapsed}')
    # Spot info
    if desc.get('EnableManagedSpotTraining'):
        billable = desc.get('BillableTimeInSeconds', 0)
        print(f'Spot training: enabled')
        print(f'Billable time so far: {billable}s')
elif status == 'Completed':
    print(f'Output: {desc["ModelArtifacts"]["S3ModelArtifacts"]}')
    duration = desc['TrainingEndTime'] - desc['TrainingStartTime']
    print(f'Duration: {duration}')
    if desc.get('EnableManagedSpotTraining'):
        billable = desc.get('BillableTimeInSeconds', 0)
        total = desc.get('TrainingTimeInSeconds', 0)
        savings = (1 - billable / total) * 100 if total > 0 else 0
        print(f'Spot savings: {savings:.0f}% (billed {billable}s of {total}s)')
elif status == 'Failed':
    print(f'Reason: {desc.get("FailureReason", "Unknown")}')

Job:    gnn-sdm-production-2026-05-05-06-32-21-658
Status: Completed
Output: s3://km-cas-datalake/processed/gnn_training_output/gnn-sdm-production-2026-05-05-06-32-21-658/output/model.tar.gz
Duration: 13:10:07.800000
Spot savings: 60% (billed 15651s of 39185s)


### 4. Download results

In [5]:
import tarfile
import os

desc = sm_client.describe_training_job(TrainingJobName=JOB_NAME)
assert desc['TrainingJobStatus'] == 'Completed', f"Job not completed: {desc['TrainingJobStatus']}"

model_data = desc['ModelArtifacts']['S3ModelArtifacts']
print(f'Downloading from: {model_data}')

local_tar = '/tmp/model.tar.gz'
s3.get(model_data, local_tar)

output_dir = 'gnn_training_output'
os.makedirs(output_dir, exist_ok=True)
with tarfile.open(local_tar) as tar:
    tar.extractall(output_dir)

print(f'Extracted to {output_dir}/')
print(os.listdir(output_dir))

Extracted to gnn_training_output/
['gnn_sdm_results.csv', 'model_Aquilegia_alpina.pt', 'model_Cypripedium_calceolus.pt', 'model_Drosera_rotundifolia.pt', 'model_Fagus_sylvatica.pt', 'model_Gladiolus_palustris.pt', 'model_Phragmites_australis.pt', 'model_Picea_abies.pt', 'model_Potentilla_erecta.pt', 'model_Pulsatilla_vulgaris.pt', 'model_Senecio_inaequidens.pt', 'model_Stipa_pennata.pt', 'model_Trifolium_pratense.pt', 'suitability_scores.npz', 'training_results.csv']


### 5. Load and inspect results

In [6]:
import numpy as np
import pandas as pd

# Find the results CSV (different name per mode)
csv_files = [f for f in os.listdir(output_dir) if f.endswith('.csv')]
results_file = csv_files[0] if csv_files else None
print(f'Results file: {results_file}')

results = pd.read_csv(f'{output_dir}/{results_file}')
print(f'\nSpecies trained: {len(results)}')
print(f'\n--- Mean metrics ---')
metric_cols = [c for c in results.columns if c.endswith('_mean') or c == 'val_auc']
print(results[metric_cols].mean().to_string())

# Suitability scores
scores = np.load(f'{output_dir}/suitability_scores.npz')
print(f'\nSuitability scores for {len(scores.files)} species')

Results file: gnn_sdm_results.csv

Species trained: 3751

--- Mean metrics ---
val_auc           0.848100
auc_mean          0.877113
accuracy_mean     0.855615
precision_mean    0.825740
recall_mean       0.523171
f1_mean           0.624746
mcc_mean          0.574949
kappa_mean        0.547282
tss_mean          0.489601

Suitability scores for 3751 species
